In [1]:
import pandas as pd
import numpy as np
import os

from category_encoders import TargetEncoder
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
from mlflow import log_metric, log_param, log_artifacts
import mlflow.catboost
from mlflow.models import infer_signature
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from dotenv import load_dotenv
load_dotenv(override=True)


/Users/manjakaranjatoson/anaconda3/envs/env_ppml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
import os
MLFLOW_TRACKING_URI = os.environ["MLFLOW_TRACKING_URI"]

In [3]:
MLFLOW_TRACKING_URI

'https://stoneray-ppml-mlflow.hf.space'

In [4]:
from random import random, randint

mlflow.end_run()

# Set tracking URI to your Hugging Face application
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

if __name__ == "__main__":
    # Log a parameter (key-value pair)
    log_param("param1", randint(0, 100))

    # Log a metric; metrics can be updated throughout the run
    log_metric("foo", random())
    log_metric("foo", random() + 1)
    log_metric("foo", random() + 2)

    # Log an artifact (output file)
    if not os.path.exists("outputs"):
        os.makedirs("outputs")
    with open("outputs/test.txt", "w") as f:
        f.write("hello world!")
    log_artifacts("outputs")

## Passons à notre modèle

In [30]:
df = pd.read_parquet("/Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/code/01_ETL/Bronze/00_Manja_test/output/parquet/df_1.parquet").copy()

print("Shape initiale :", df.shape)


# 1. Préparation de la cible + nettoyage de base

TARGET = "arrival_delay_min"

# drop les lignes sans cible
df = df.dropna(subset=[TARGET]).copy()

Shape initiale : (83948, 74)


In [31]:
# 2. Feature engineering dates / heures

# Conversion en datetime de ces colonnes :
datetime_cols = [
    "flight_date",
    "movement_date",
    "scheduled_departure",
    "scheduled_arrival",
    "time_dep",
    "time_arr",
]

for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True).dt.tz_localize(None)

# Features depuis flight_date
if "flight_date" in df.columns:
    df["flight_dayofweek"] = df["flight_date"].dt.dayofweek
    df["flight_day"] = df["flight_date"].dt.day
    df["flight_month"] = df["flight_date"].dt.month
    df["flight_weekofyear"] = df["flight_date"].dt.isocalendar().week.astype("float")
    df["flight_is_month_start"] = df["flight_date"].dt.is_month_start.astype(int)
    df["flight_is_month_end"] = df["flight_date"].dt.is_month_end.astype(int)

# Features depuis scheduled_departure
if "scheduled_departure" in df.columns:
    df["sched_dep_hour"] = df["scheduled_departure"].dt.hour
    df["sched_dep_minute"] = df["scheduled_departure"].dt.minute
    df["sched_dep_dayofweek"] = df["scheduled_departure"].dt.dayofweek
    df["sched_dep_month"] = df["scheduled_departure"].dt.month

# Features depuis scheduled_arrival
if "scheduled_arrival" in df.columns:
    df["sched_arr_hour"] = df["scheduled_arrival"].dt.hour
    df["sched_arr_minute"] = df["scheduled_arrival"].dt.minute
    df["sched_arr_dayofweek"] = df["scheduled_arrival"].dt.dayofweek
    df["sched_arr_month"] = df["scheduled_arrival"].dt.month

# Durée programmée du vol en minutes
if "scheduled_departure" in df.columns and "scheduled_arrival" in df.columns:
    df["scheduled_flight_duration_min"] = (
        (df["scheduled_arrival"] - df["scheduled_departure"]).dt.total_seconds() / 60
    )

# Période de la journée départ
if "sched_dep_hour" in df.columns:
    def get_period(hour):
        if pd.isna(hour):
            return np.nan
        hour = int(hour)
        if 5 <= hour < 12:
            return "morning"
        elif 12 <= hour < 18:
            return "afternoon"
        elif 18 <= hour < 22:
            return "evening"
        else:
            return "night"

    df["period_of_day_dep"] = df["sched_dep_hour"].apply(get_period)


# 3. Drop des colonnes de fuite / inutiles


cols_to_drop = [
    TARGET,                    # la cible doit sortir de X
    "retard arrivée",          # doublon / fuite
    "arrival_advance_min",     # très liée à la cible -> fuite potentielle
    "flight_date",             # remplacée par variables dérivées (voir code ci-dessus)
    "movement_date",           # idem
    "scheduled_departure",     # idem
    "scheduled_arrival",       # idem
    "time_dep",                # idem
    "time_arr",                # idem
    "__index_level_0__",       # index parasite parquet
]

if "status" in df.columns:
    cols_to_drop.append("status")

df_model = df.drop(columns=cols_to_drop, errors="ignore").copy()

print("Shape après FE + drop :", df_model.shape)

Shape après FE + drop : (81098, 80)


In [32]:
# 4. Séparation X / y
X = df_model.copy()
y = df[TARGET].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [33]:
# 5. Target Encoding


high_card_cols = ["airline"]

# airport_destination n’a que 6 modalités, donc pas high-card,
# sinon pour être comme Patrick et Ludo tu pourrais ajouter :
# high_card_cols = ["airline", "airport_destination"]

high_card_cols = [col for col in high_card_cols if col in X_train.columns]

te = TargetEncoder(
    cols=high_card_cols,
    smoothing=10,
    min_samples_leaf=5
)

if high_card_cols:
    X_train = te.fit_transform(X_train, y_train)
    X_val = te.transform(X_val)

    # on supprime les colonnes originales comme dans le code de Pat et Ludo
    X_train = X_train.drop(columns=high_card_cols, errors="ignore")
    X_val = X_val.drop(columns=high_card_cols, errors="ignore")

    print("Target Encoding terminé sur :", high_card_cols)
else:
    print("Aucune colonne high_card trouvée pour Target Encoding.")


# 6. Détection auto des features catégorielles

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_features = [col for col in categorical_features if col not in high_card_cols]

print("Categorical features finales envoyées à CatBoost :", categorical_features)
print("Nombre de features catégorielles :", len(categorical_features))


# 7. Gestion simple des valeurs manquantes sur colonnes catégorielles

for col in categorical_features:
    X_train[col] = X_train[col].fillna("missing").astype(str)
    X_val[col] = X_val[col].fillna("missing").astype(str)

# PS : Selon ce que j'ai lu, CatBoost gère bien les NaN sur 
# les numériques, donc pas besoin d’imputer tout de suite.

Target Encoding terminé sur : ['airline']
Categorical features finales envoyées à CatBoost : ['flight_number', 'airport_origin', 'airport_destination', 'terminal_departure', 'terminal_arrival', 'movement_type', 'icing_conditions_dep', 'freezing_rain_dep', 'thunderstorms_dep', 'has_precipitation_dep', 'fog_dep', 'icing_conditions_arr', 'freezing_rain_arr', 'thunderstorms_arr', 'has_precipitation_arr', 'fog_arr', 'GREVE_LYON', 'GREVE_TOULOUSE', 'GREVE_NICE', 'GREVE_MARSEILLE', 'GREVE_CDG', 'GREVE_ORLY', 'period_of_day_dep']
Nombre de features catégorielles : 23


In [34]:
# 8. Pools CatBoost

train_pool = Pool(X_train, y_train, cat_features=categorical_features)
val_pool = Pool(X_val, y_val, cat_features=categorical_features)

# 9. Modèle final

model = CatBoostRegressor(
    iterations=10000,
    learning_rate=0.045,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    early_stopping_rounds=500,
    verbose=200,
    task_type="CPU",   # pas de gpu avec catboost sur mac..... donc on se met de mon cpu
    l2_leaf_reg=5,
    random_strength=1.0,
    bagging_temperature=0.7
)

print("\n=== Entraînement CatBoostRegressor ===\n")

model.fit(
    train_pool,
    eval_set=val_pool,
    use_best_model=True
)


=== Entraînement CatBoostRegressor ===

0:	learn: 26.9409441	test: 26.2013136	best: 26.2013136 (0)	total: 78.5ms	remaining: 13m 5s
200:	learn: 18.8323811	test: 18.4354971	best: 18.4354971 (200)	total: 10.3s	remaining: 8m 21s
400:	learn: 16.6966612	test: 16.7604544	best: 16.7604544 (400)	total: 19.8s	remaining: 7m 54s
600:	learn: 15.2079823	test: 15.5893425	best: 15.5893425 (600)	total: 29s	remaining: 7m 34s
800:	learn: 14.0044969	test: 14.6586070	best: 14.6586070 (800)	total: 37.8s	remaining: 7m 13s
1000:	learn: 13.0544404	test: 13.9398123	best: 13.9398123 (1000)	total: 46.6s	remaining: 6m 59s
1200:	learn: 12.2942222	test: 13.3620254	best: 13.3620254 (1200)	total: 55.5s	remaining: 6m 46s
1400:	learn: 11.6278769	test: 12.8633850	best: 12.8633850 (1400)	total: 1m 4s	remaining: 6m 34s
1600:	learn: 11.0077276	test: 12.4095041	best: 12.4095041 (1600)	total: 1m 12s	remaining: 6m 22s
1800:	learn: 10.4833146	test: 12.0390968	best: 12.0390968 (1800)	total: 1m 26s	remaining: 6m 31s
2000:	learn:

CatBoostRegressor(bagging_temperature=0.7, depth=8, early_stopping_rounds=500, eval_metric='RMSE', iterations=10000, l2_leaf_reg=5, learning_rate=0.045, loss_function='RMSE', random_seed=42, random_strength=1.0, task_type='CPU', verbose=200)

In [35]:
# 10. Évaluation du modèle de Patrick et Ludo avec dataset Philippe et Manjaka

preds = model.predict(X_val)

mae = mean_absolute_error(y_val, preds)
rmse = np.sqrt(mean_squared_error(y_val, preds))
r2 = r2_score(y_val, preds)

print(f"\nMAE   : {mae:.3f} minutes")
print(f"RMSE  : {rmse:.3f} minutes")
print(f"R²    : {r2:.4f}")
print(f"Best iteration : {model.get_best_iteration()}")


MAE   : 3.451 minutes
RMSE  : 7.958 minutes
R²    : 0.9101
Best iteration : 9998


In [37]:
import mlflow
import mlflow.catboost
from mlflow.models import infer_signature
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mlflow.set_tracking_uri("https://stoneray-ppml-mlflow.hf.space")
mlflow.set_experiment("ppml-catboost_arrival_delay_regression")

while mlflow.active_run():
    mlflow.end_run()

preds = model.predict(X_val)

mae = mean_absolute_error(y_val, preds)
rmse = np.sqrt(mean_squared_error(y_val, preds))
r2 = r2_score(y_val, preds)
best_iteration = model.get_best_iteration()

signature = infer_signature(X_val, preds)

with mlflow.start_run(run_name="ppml_catboost_regressor_arrival_delay") as run:
    mlflow.log_metric("MAE", float(mae))
    mlflow.log_metric("RMSE", float(rmse))
    mlflow.log_metric("R2", float(r2))
    mlflow.log_metric("best_iteration", int(best_iteration))

    mlflow.log_params({
        "model_type": "CatBoostRegressor",
        "target": "arrival_delay_min",
        "iterations": 10000,
        "learning_rate": 0.045,
        "depth": 8,
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": 42,
        "early_stopping_rounds": 500,
        "task_type": "CPU",
        "l2_leaf_reg": 5,
        "random_strength": 1.0,
        "bagging_temperature": 0.7,
        "high_card_cols": ",".join(high_card_cols) if "high_card_cols" in globals() else "",
        "categorical_features_count": len(categorical_features) if "categorical_features" in globals() else 0,
    })

    mlflow.catboost.log_model(
        cb_model=model,
        name="catboost_model",
        signature=signature,
        input_example=X_val.head(5),
        registered_model_name="CatBoost_Arrival_Delay_Prediction"
    )

    print("Run ID :", run.info.run_id)
    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"R2   : {r2:.4f}")
    print(f"Best iteration : {best_iteration}")

/Users/manjakaranjatoson/anaconda3/envs/env_ppml/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run ppml_catboost_regressor_arrival_delay at: https://stoneray-ppml-mlflow.hf.space/#/experiments/3/runs/f20dbdc47ff7413eb1e8b80880d20e5e
🧪 View experiment at: https://stoneray-ppml-mlflow.hf.space/#/experiments/3


S3UploadFailedError: Failed to upload /var/folders/p7/7w66qcfn4yj82k3sbs5fp74h0000gn/T/tmppl0qtwec/model/python_env.yaml to ppml-mlflow/artifact_mlflow/3/models/m-25ac71610240422d9ad250bdd92e209c/artifacts/python_env.yaml: An error occurred (InvalidAccessKeyId) when calling the PutObject operation: The AWS Access Key Id you provided does not exist in our records.